# K-12 School Districts
California's roughly 1,000 school districts spend about half as much as all cities and counties combined. They report to the Department of Education and the Census Bureau rather than the State Controller, so they come from a separate source: the Census **Annual Survey of School System Finances**, fiscal years 1992-2024 (FY2024 = school year 2023-24).

**Read this first**
- **State aid dominates and is not new money:** roughly half of district revenue is state aid (mostly Proposition 98), which is already counted in state spending. Do not add it to state totals; the burden page uses only the districts' own local taxes.
- **Local property tax:** the "Local property tax" layer is what districts collect directly. Part of it exists because the state shifted property tax from cities, counties and special districts to schools in 1992-93 (ERAF) and later years, so the local mix reflects state decisions.
- **Break at FY2019:** Census reclassified over 900 California charter schools (those whose charters are held by non-governmental entities) out of the school systems it surveys, so district revenue, spending and enrollment all step down that year. Marked with a dotted line. Per-pupil lines also jump there: enrollment falls by more than spending, and Census notes the state raised its contributions to school employee pensions that year.
- **FY1994 is missing:** the Census file for that year covers only 258 of about 1,080 districts, so it is left out and lines cross the gap.
- **Not included:** community colleges (also Proposition 98) are not in this survey. County offices of education are included as districts.
- Totals follow Census' published state tables: transfers between school systems are excluded from both revenue and spending. Fiscal year ends June 30; the survey's FY2024 is the school year ending in 2024.

In [1]:
import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
from ca_taxes.io import PROCESSED
from ca_taxes.deflate import add_views, base_fy, LAST_ACTUAL_FY
from ca_taxes.style import stacked_area_toggle, base_layout, PALETTE, SOURCE_NOTE
pio.renderers.default = "plotly_mimetype"
SCHOOL_NOTE = "Source: US Census Bureau, Annual Survey of School System Finances (California districts)"
BREAK = {2018.5: "Charter schools reclassified"}
sf = pd.read_csv(PROCESSED / "school_finance.csv")
enr = pd.read_csv(PROCESSED / "school_enrollment.csv").set_index("fy").enrollment
rev, sp = sf[sf.kind == "revenue"], sf[sf.kind == "spending"]
stacked_area_toggle(add_views(rev), "group", "K-12 school district revenue by source", SCHOOL_NOTE,
                    order=["Local property tax", "Other local taxes", "Other local revenue", "State aid", "Federal"], breaks=BREAK).show()

In [2]:
stacked_area_toggle(add_views(sp), "group", "K-12 school district spending by function", SCHOOL_NOTE,
                    order=["Instruction", "Support services", "Other current spending", "Capital outlay", "Interest and other"], breaks=BREAK).show()

## Per pupil
Spending and revenue per enrolled student, in constant dollars (fall enrollment as reported in the survey). Enrollment peaked around 2005 and was roughly flat to 2018; the drop in FY2019 is mostly the charter-school reclassification, not lost students.

In [3]:
mac = pd.read_csv(PROCESSED / "macro.csv").set_index("fy")
t = sf.pivot_table(index="fy", columns=["kind", "group"], values="amount_k", aggfunc="sum")
f = mac.cpi_factor.reindex(t.index)
def per_pupil(s):  # $ thousands -> real $ per student
    return s * 1000 * f / enr.reindex(t.index)
tot_rev, tot_sp = t["revenue"].sum(axis=1), t["spending"].sum(axis=1)
fig = go.Figure()
for i, (name, s) in enumerate([("Total revenue", tot_rev), ("Total spending", tot_sp),
                               ("Local property tax", t["revenue"]["Local property tax"]),
                               ("State aid", t["revenue"]["State aid"]),
                               ("Instruction spending", t["spending"]["Instruction"])]):
    fig.add_trace(go.Scatter(x=t.index, y=per_pupil(s), name=name, mode="lines+markers", line=dict(color=PALETTE[i]),
                             hovertemplate="$%{y:,.0f}"))
fig.update_layout(yaxis_title=f"{base_fy()} $ per pupil", yaxis_tickprefix="$", xaxis_title="Fiscal year (ending June 30)")
base_layout(fig, "K-12 revenue and spending per pupil", f"Real {base_fy()} dollars. " + SCHOOL_NOTE)
fig.add_vline(x=2018.5, line_dash="dot", line_color="grey", annotation_text="Charter schools reclassified", annotation_position="bottom left")
fig.show()
fig = go.Figure(go.Scatter(x=enr.index, y=enr / 1e6, mode="lines+markers", line=dict(color=PALETTE[0]), hovertemplate="%{y:.2f}M"))
fig.update_layout(yaxis_title="Students (millions)", xaxis_title="Fiscal year (ending June 30)", showlegend=False)
base_layout(fig, "K-12 enrollment", "Sum of district fall enrollment. " + SCHOOL_NOTE, height=420)
fig.add_vline(x=2018.5, line_dash="dot", line_color="grey", annotation_text="Charter schools reclassified", annotation_position="bottom left")
fig.show()